# Per-store window features with exact missing-value rules

A model that forecasts daily sales needs **features** computed from each store's past: yesterday's sales, a short moving average, the average so far, and the store's share of the day. The rules for missing values are part of the task.

```python
def make_features(df):
```

`df` is a DataFrame with exactly three columns:

- `store` (str): the store name,
- `day` (int): the day number,
- `sales` (float): that store's sales on that day; `NaN` when the value is missing.

There is at most one row per `(store, day)` pair. The rows are in no particular order, and a store may have no row at all for some days (gaps).

Return a **new** DataFrame with exactly these columns in this order: `store`, `day`, `sales`, `prev_sales`, `roll3_mean`, `cum_mean`, `day_share`. Its rows are sorted by `store` (ascending, Python string order), then by `day` (ascending), and its index is `0, 1, ..., n - 1`. `store`, `day` and `sales` are copied unchanged. Do not modify `df`.

For a row $r$ of store $s$ on day $t$, where "the rows of store $s$" means that store's rows sorted by day:

- `prev_sales`: the `sales` of store $s$ on day $t - 1$. `NaN` if store $s$ has no row for day $t - 1$, or that row's `sales` is missing.
- `roll3_mean`: the mean of the **present** (non-missing) `sales` values among $r$ and the (up to) two rows of store $s$ just before $r$. These are rows, not days: across a gap, the previous row may be several days earlier. `NaN` if fewer than 2 of those values are present. Rounded to 4 decimals.
- `cum_mean`: the mean of the present `sales` values among $r$ and all earlier rows of store $s$. `NaN` if none is present. Rounded to 4 decimals.
- `day_share`: $100 \cdot \text{sales} / T$, where $T$ is the sum of the present `sales` of **all** stores on day $t$. Rounded to 2 decimals. `NaN` if the row's `sales` is missing or $T = 0$.

**Example.**

```python
df = pd.DataFrame({"store": ["B", "A", "A", "B", "A", "A"],
                   "day":   [2, 4, 1, 1, 5, 2],
                   "sales": [15.0, 30.0, 10.0, 5.0, 20.0, np.nan]})
make_features(df)
#   store  day  sales  prev_sales  roll3_mean  cum_mean  day_share
# 0     A    1   10.0         NaN         NaN      10.0      66.67
# 1     A    2    NaN        10.0         NaN      10.0        NaN
# 2     A    4   30.0         NaN        20.0      20.0     100.00
# 3     A    5   20.0        30.0        25.0      20.0     100.00
# 4     B    1    5.0         NaN         NaN       5.0      33.33
# 5     B    2   15.0         5.0        10.0      10.0     100.00
```

Row 2 (A, day 4): A has no day 3, so `prev_sales` is NaN; the rolling window is A's rows for days 1, 2, 4, with values 10, NaN, 30, so its mean is 20.

**Restrictions.** Only `numpy` and `pandas` may be imported.

**Constraints.** Up to 50 stores and 20,000 rows; days are between 1 and 1,000; present sales are between 0 and 1,000. Rounded columns are checked to the stated number of decimals. Time limit 2 seconds on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem6_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem6_submission.py
import numpy as np
import pandas as pd


def make_features(df):
    # Your code
    pass


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem6_submission as submission
importlib.reload(submission)
# e.g. submission.make_features(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XAt32zaW/isY9ZwJmcqM7DSZVq2766RuJ3PyOklmZ/ZYPiwlQjLHFKkhKD/q8f72/b4LkAQl2Xl0dtPTiASBi3sv7htAbgZZOhirwXr03d5q9HQwVIOlrhM03Wx/qbM612x7q6s9U5eVVpdZkZaXaq6Tel1pg/f6TOmrZFarZWZMViz2LpJ8rVW1zrUhEPyYsgCUYp3nnC4r1jU+jdXjEV7TbD7PZuu8vkbLN2jQRV3xebBMznXcTCSIlimAxkWyFKRWVTnN9fJpbNZTmRuzoFeS5+WlTuNsuSqrmvOcDIr1cnXNj6ukSBMzOMXjNCkKTYpP3Fucl+WK/edJbjTJ16aO62ypy3Ud88MBEa6ThQVqYUWLqlyvph70yPLIa1jqaqH9d8spr8UAVcHKlOtqJuSVVbbIiiS3var6AI11tda3eJ9nuXDwZnCWVAUYHK2EZRP57886X+nKqPosqZXjklpUScpGyxYV/DqvyqVyw5vWh7+G0aSYFEeuu8qMmpVFjTV5tC6y2jz6gT8/PnJQ8b7K0h8f2d5AIlIvapXqeQaoivxT83Uxq7E0RnHdUmmMH1rcaqwwnvSk6JZQ2VVWv6Lp10gd//3o1duXx+9VnpmaJGkLlsCMMmflZaEw6N26UMFFZjJgFX6v3hNcDRnEtPpCV9cyCKT9nGS5yG2VZEYrvloRThTgmWQBCa+yGgSreVmpZ7oYK5Ncq0uie5lA3quyWAxVQagqhVipsyxN0T0rVuvakHt2FQrH0lm5um5fKEzkr3sVsVSAWqzY+pV6XubJVD1/+1ejEiiboSRX5FVBVB4Y9epAva3KoTKlystZkhMiJMmo5RpMmWoyFSwqqVMZPucZ+TAF/WdYyjnUtKwiznQQjbi4iVqsuf6BnZlSmeu9+qzSSarMSmPB0AsSvjjLryEt+VwBF2AxgzEAoytNptcqmxNBvIrOOTqc6pAp798eH/8U/3z0/MObd+qQs5PgSTHLE8zOZQiOr2Z6RUkJx5NC4c8Kn2wvCBQ4rtMAspgO1dIsmj6YtyjJ5CJ1LfzTLW7Avh2QuFgFV81YmBlvTLNCZTU781rnoD8rTJ0UMx1cDe336IMuoLGhN1ym1TBVhbqKUljU2VkQRrPVGn/LMgeh7ayFTPVCpjuuqrLyoFiSPVjFKkpMUlXJNfDu6DBnyUoHi7IeQigL/g35PJwMINmwpJNBg5kBr+v1KtcBKUf/MJKhDhfhKfo0nQgrHKr5ZHBDgLdYcGPnUjcGdkdfrfSshkzceP1vMV2H2SwvzX2YDVVVl/nhvt57OlSJe/y2wReadQmUW2QTU1+D0HleJkSM7TJn/4NHzMLSR5Iu7eMd5FgCXPdwF23NJyGvnQALkhmYt6zGaOCR50GoYCkohf63S/fNn56WNMnEEhYckxXzFrgTZK53njsmghmOXfzLsYt/+YKnqwocE0YQuWVyxZ9kaoKF2lOXYRgSNgjNftNKw7GpEZVvh6r0+SSGTokvh3nIE3gwWBhNgVU3+BlHjxe3YW/t9T/XSf4JUtlXKtuVSMMTUtKHoXAUSCR1Xbnvk4EsRwdkh7h4AtL1+T8XC180BP/Y8QHIbSmTz9UWAldlvIkxhIHYCvEdDBjjG3z6Q9VDjb3Q1FuMzMSiIrIcfDJ3rodM6K1IN2QLf7aCFQ5wGMUSjcVxD50FzB9Xzs36oOn0oIHaxzQ2RbKCJ6//f01zXhb6M40ypmyk8ooiKb59EMIlpqK9fXyCNJuBkQxcQh+fPnE97AgwSLVeHX5ApOcJmEPwA9i3id42gLDnQtgUEaZ8u+oxHssSJEM1/XK+J5/Ed9vF6oXM96VsT8h2gWP6psBNlNhJTDANd9kaDO/szM7hrZ2YukdZ275mk4KhtXaQ7MJfqwaOQPCClzJerqEUiG6CeTFUD2FOoRkPH55f8qnBxIaOz+EB1LyQmecMUEkDYhxEgQuoV1IgZKwW6yWCcrgQtcqTmY6aqFO8PvUJdvGk06wklHg24QDOeGp7Ijqj8yiCTYTsZw7JIMgg2IQcqhHK6CqBf/stWwV2jMzRE3CxJ61wOSNyjcQGsphqBvjZPGNkyZC5o+Wrm0x9rfZvv1eXZXXOqD4R4VVcPsSjrb10TAbyHYcZxGdFYDRDQWD1qVz+3mcxrKsfcu+IpCW5hD80iL1BARkkse6DdiSCd5tDcao3AFExBUOCtDQkSeLwV8lMzTABh4BIG3UzW5HAnsEBsou96TqFJWWzxOAIyzMmUOsV1EnVl9lMEyA4lMFDl3OnwcC54agREWLLHPES3TchYMQa3yL1gR+yCs0gJn/AKegX7GTksE57YlWPGE0CQoTsch4LGF011uYjopTWuwcjRqldOOLoPVRuEdUj5acNnUjGFMWK2hAc+HKHRcQ0Pxw6SBumaIqc5rxrSg6G6vwAs/Xto6A83GjsUfJxVgieZMSBsOHAaxY2IGMLUrrYO9jhx5s+QaJHdVmK3CD5T+txdDC/RZQPKdPfu6yrFWIRJqidAFLrgvn8jWONGyc5JBK7Zol3/JEpkVkiDZ5DtBYMYG2SeePWSQBRO+5V0KpYQDt12ixYl+BgJdNyCW7PE4hf3HacSDlnbZIkK2OpMPSqHL+wBRoAzLriQSZG4yzLU2rdjJURJMwZdYE9//L+zWvgWojmmDoFhqKrq+v6DC17L1Qzj/rBlTjiNKt+VD90c+Aj3qGV/5K2+setlB/K5qX/WNirOs+mbZP9QUu0rrO8ay7bx3+YsisSrJL6zB9uskWRdMPMtdkoLzQvFfzCNJmdty2Q4oLFAltrMLMMZBa6NXdoOM/qvVyjm9qPnkb7LqYzY9aJlizkDcWkILtwzqcxfqskA5cXk6KZI5pnOSS6eQ0mA6BdVpoZoKu0IBKNHv61OC9YwzFlzopKKRUATJituGrRQxcp/vn43TE0x/EieovfII5ZBovjMILh4nBEdaukAmaTAlyJ2DmCtOqqDkZQBoQPBBMC4K7al9SC1Ffw1f9Mxur4m9GBVMJe/u3ov9/Hz45evz7+CRjcTAaxKy3GMYkBYc162ld9pWfuCXG+fZqVyxVwnQxuJ8XLN2/exq/f/HT8HuACyEr0cwld5cPfILjaPh6Z62LWfniJGPI5YNi399p7+QlBZvf2iy7on8vq+GoV+iWWD7YW87EqS5wg01sGlLH1EvYGbku3GiupYgPIr0YwspnFszM9OzeBqWZc4jrp+9t3Vt8ZDcNZkY/luWXO0izAmO9hFhkB2hqfqI2pH7SrI8GQKyXSUdVVZsuKPRdlndDJ6a5otq40AjvhEqQE+TUQ7ceh76+Rn19JHMqinN6OD08c2mNIi9SIHfZjmkkjw12KXBbWztzoiL9FeTvmM3oj+bl1CLpSNV0C+BXBMENPNurX5NHJqUN0mrTt4ifrxuVIOTReGwHmtU+hD7Gtlbp2ijhCRuhulkJVGIlVNlCwgAHB9g+Y0TDozqas14Occ4nddB5RsN0EUjP3eL5M6uU691vmUh2BqaZtBvMvk/w84GJsuG4vTGd3K882NdjMKtpQlh0jQXe87cBcSQXxcpBIJwblS5YW7GuE2cBIRo7BEpEnzBGTOQJfCrJ2IcLxbt/orUWUpKmbI9zu3K2N1y8yqzzjcnPOk7390x0Du8WL1quUUfeOsX6SmN/Pxp9h9TapwYJirYSPrvAO8juF2sFLGeEYieffwcVNDgLaBhc81vU4sEsCNobexbwvGip0fi7Tjxrd2aR9QyAEDyraJ0F97Vnk+wBm6d1U7e60a8bOWW3OKYofJauVLtKgqfjYQZOBNXnWvP9Hn1+7ZhEv+Cwr3jgvdrReHBn6oNDWVjYGRKXr9yqpXyFk3JJpa4M+Ezkg5gmkB5JBogN1p/lvogCrJVbk2SoKwX7RP0qmp2JgA2+eMNxtXb6mRwFjGlA3D4bqgQXSaBQV8AHmQ967eMBS7O3ueuIu/Fmy2IG+aXDvgH3V7iBOk8KQt7Mz5yEkFOwchJSfdYIk9u/yG35vs9H8Mrk2e3a7FZkM4sKsBUShdNDKIr8eNnMic+lcTWClwBaRXCjFoONa5XqB1PySYZOSnlEUhZ4zQ1Cm0pKLYgGfZZJf2lUIPKf5R3GOnhe26DrnCzH8lwo8X/pH1YsLPRHCDJ8tOsDAKMcfwg9MuEN2ADn8PStclM0c7ZZst8rAvE98bJUoLlc2nuxX29jdKtmG6n0KwcwoIRf/qRrYTFk9vVGBjZzcDCejU1+4d5D/OSzoZr2PeDFuWyRL65cRbMMkEfYeoWimGlOoK32mC0li29hRRv176XeU3d6Tqc+rDGDy61gi2QAgkOjGzKb6Ef2bQu8Jnqk2syqTrEKtIKrQ74VyO9HSIWu2r5mwQV01+xnZrO9XmKYsqzR5a4S8Gc91XE8DHcVx+wEpnws2CRyR5lyc+pzzAARWas70UzcxHzO/lgQXlV6yZMKCFa2s1Cad0OFvhmNe1N5r4lKIwLC3bKd16DtmcsdEdki0tz/C6P/2Rma9Zf+G2UtE4kGbi8HSbma5zGSTanFx0oaIDSl39jw49VJixykG4Ieq7fK4Cdch92hnwSHKyyQ1QSBIPKLYUCekFDEgPPgsVjKCxtghfSJEh4zfoeGGVBMP1bZe2IRx4s7eTOSgjelebWHGiqzLpCmJbHhdYgl8+CcdtFOic0cq2io6I1iqoDmxWJ2Kt+CbaMmpH842KV0TzxqpO25N61sJqVoEwk4eDjGB7e57eSslTclzDqSzMnpfs3zy4k3DPFvkiexP4N7ev/jl6OW7V0OXpbuuMGexFHebXWDPlvVPMJGXB6MGl352LIdgumIVljPNKg3ls4sRANPNMKvBUdcZp6gaNF98ePHq+F387vjo5bDFbiPMMStN6emXwiK2xqzN2MoOq5OyZeJR5J0Bs6LRFeawTJ6p2phvPd2ezsGSCTl1wL82B0JjbD9zws+SMJxawd8mSXRIVxFrQLEdRssTfgHjRv3ahCu79EoSThCdcpxaa2ZppAUWqyYWty7Lc9lTUDfNeoyjESvAjelyszxLjG4rRLYG0pXEnr08Ho32OwQ+i4y7MHYbQ4Imd2J4Qqx0C+WCoHv8UavWW6B3MaqxK5wX8kyJkp34AAn4N6PRaHz6xYoshxb7dRxp8s932D7NXnGzowlahvZT+PHF7XiVltoGTfawHbwLQdz6eftnkrD4PJ2cDOyhP2ph6zLac4At3e4c4cd1b+Ep3+IuXbLQXC933M8Vtposz3ahZWiODjZBvDN79DUMGgIkKnNr2PlE236RVKaZI8pqvTSBnJ4p+l5BrGqz788NlARoBPPQL3jBSnATUkISzrgRPIs7pvRXa9gtKxSMVuQUQdGQtmFwaZ6zYq399ZtZB2vt4ViAnDwZn8J+y74wsHVVaiXmcTJwoJvO3mz8ymow055daYqzt3bTh02jaHT7OXti2wcfvsDr/F7P4zbntq3yF5k0twgnHeeoqQy8t89ubFnwbrCNPayOi7HOy2LR7Mts2mwVZIU9ZSbR/JAJxDypVLMzGPpGwE0umxsbNe3fSXCLM8PrHWdVPsuX/Ltw8c8ZyH5F+ok+pIPVCjjhVZDiNLhro3aoHu9yby6cPW0Ssqo1bJ/titoBNlY9ZOoYdLHrfVGpGJVpWebBFmL2G0HVPckluLoHrh3RJCh3uhQ51jxXTcJjrVscM7/hRtm42R5gumN3dltnwT1dVz5Dzn921wHpdj+UJ+bZuErv3skbyq730D95M3SHRIbci+Yk7dlyeANn1fVVslzROoL/z9+8dJ/k6oM1f2ly7exgkmtjH5G6X8Tee1Xm+eOYJTC3+7deem+AEBtgbCf56fj5i1dHMtFNf6TchvAHuwZv/Fgd3Ho77HqO7JJF0qq83Djt8pY7Pntv7TZ321EJ9wIhb6gAGGpBMvbKao+ZVmjPBZtI/Zc9B8oCxroQpdg4HDIFUi2YuC7rJCdJt0P837lFU+P7UF2IjAFJzxo1AKj/7ihAwO7w3I0eBRy6UUu+aIpHxHfDuLWYnKRUnfZVAjTAgv8KYRsudu0dWmyJZ1Ozdfj5/ig3Xjmx6XBian8zpyxiTMzps1kd5D1/BJWXUOTCHmtxjEEfLxBtzmAJ7f0zWAS25VALArySYVc2psnNCU8Fw1plsFoH4ViOWZ2Se1c+9043zLHWO2F94vCaB1h38/y+ypVTtbGIilO3MWWm1bixuthUurHjsp0ERO6HwzsPtKhNNTPrZQC+hYhgc13YR1b7mucfD9WBLfWQzHsB+8pKsORhA9c+A7Aw9tPg9XR9fzRSDyEij4S3m7LvSuS1+sOhGnXg7yv01WVMJQzS+cahnJPAQFhg6VOEjAIcs63SKDNFEkAKBbotNkAfra50is2jgem8M5un+MRXWcvmxa3cKYPm9u4AqzY2IUrnQ4WwWrdH3a12+uco0S/qXRHjqHD3iWaMHpKAn5I6+VnONNiDzTLF7bg7ttl2GCqewLZ1PIz2Knn9+wA8z8AO0azM18vChHR89Bz9CdzX5pqOvTGXX6sbdr11k23B6ubiCW+GI62J99auhw4EjegSCxHg7kZHg8pN0+dWTLB/cLsdcReRGdbpygJnkz2D144KN2Zi/VdGtGTDBu0PVRRFcMxU093TeKIjc51cei0ibZeUM85py3cnlNWrsDNUFogVuNMWhH3fAODrX08keM6zKdhbE897VM7J1axhCzzvqCyWTc5EYUlPDsZ+CUJO8wtW6HMaYfHsnaCU8nXo32KxAfRKCcrsvIFu14tVL3bLWhexwFruOKYrx/kB094EWcqpKChyESxCGotA04yIuz/dvNFQ2ludGwIEtOxtAHePsb0l8jp5LfNz4e1NB+FiIILFzk7mwp2XJ7p7NWV98j/SIFiWdXjau03TR6S9VeNfp+Gfc3CnCa/EM6C793ma8OTDqO9pycLGhrV8G2+dhWhZtiM53c7S3aDzewaVRLa5tINF/OFQ7eu97/odd2z7uAhjUWVpC+AhPIV6+BDz7bn8xWuDEPxA0E/vwKEB5o75efiMoictmL1zxgBbCJKlXx+KxpTnGwss7D6kBdgtS737RliBGwyw0iIJXCAbNUIOC4Q357cq1bNsiYjE7cicd9sxYejvZnFH1Vq66lDE3PEwZe3ddwqul7O+h/cH/k2M1/NmjHj891331FitvCNoFYzSeZTl5eykYsbZejtqEG09KzuIkGOxq0Falasd1w/SuefkvbRGqh5uPs/ZWhadBJPBM9kigJF+gjANFmUyOGIL8o7Ho14LrPh+2/LMtfQHPeFmQ6/lwN6+SArocy/s4aFJl3eFG5gvkpUBsYzujI++GBZW8SaDvxM2go9+QNJEIycBMPuGWGD6p/wFYt/y9ylp4MN3QwaMKtiXhP709A4GieWiBBAngiZOY9UFoko28hnt75EdtIKMNGkh3bV56WBlepNQZ0tjqwEk2Y65i2rjqL7YQa7jsqX5sV2Dx37zN/4L+PEn24ccsU9/6pbqo+xovIDlASxxlmrrAiwB27T+pqvSpgVc1zsITJxUtTI03WxInFA96fU48GlDSO7a2lHS0HHjo+QlsqKXZyWMy4rKh0jMkooIn7ZohOSsidPbSAeucOiquXMW6Eb9q5ILqHOFDjZlNHyiNA3VOS9/jaJvh+Jc8bT/ZJM3XoZ63t2EaED53koEo+uy30xD473Dq1XuKH5AH0FEdjibC0bhwrmtERKQeKlBhXCx1gtdGeaeyF72dx12IlVtgj93JvfmfDw6SG9bEe+bt55J35Qtc7aez3Odxg4zX7gYQReLYN+L2eqOPU99lnjy4NaKMaZH0j6tSbir+eBJEwm7fxpAdmdvarnctK0McrU2BrIxb+n4+Io7kNmB9AFAPgEfvxmNWin57km4Jb2N+xg/Ho3uchZWrNFhD0xUswRLNtVz/vMe8u8ryD8pIMh1+Ye9XnUQjaRyemfitZHCTQZCnUwhsAa3t/8LHXrBbQ==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
